### Project Setup and Library Imports

First, we'll import the necessary Python libraries for data manipulation, visualization, and numerical operations. These libraries will be used throughout our LSTM project for data loading, preprocessing, and model development.

In [7]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

### Load the Dataset

Next, we'll load our dataset, which contains quotes and their authors, into a Pandas DataFrame. The dataset is stored in a CSV file named `qoute_dataset.csv`.

In [8]:
df = pd.read_csv("qoute_dataset.csv")

### Initial Data Inspection

To get a quick overview of our data, we'll display the first few rows of the DataFrame. This helps us understand the structure and content of our dataset.

In [9]:
df.head()

,quote,Author
0,“The world as we have created it is a process ...,Albert Einstein
1,"“It is our choices, Harry, that show what we t...",J.K. Rowling
2,“There are only two ways to live your life. On...,Albert Einstein
3,"“The person, be it gentleman or lady, who has ...",Jane Austen
4,"“Imperfection is beauty, madness is genius and...",Marilyn Monroe


### Examine a Single Data Point

Let's look at a specific quote from the dataset to understand its format before we start cleaning.

In [10]:
df['quote'][0]

'“The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.”'

### Check Dataset Dimensions

We'll check the number of rows and columns in our DataFrame to understand the size of our dataset.

In [11]:
df.shape

(3038, 2)

### Extracting the Target Column

For our LSTM model, we are primarily interested in the 'quote' column as it contains the text data we will use for next word prediction. We will extract this column into a new Series.

In [12]:
quotes = df["quote"]
quotes.head()

,quote
0,“The world as we have created it is a process ...
1,"“It is our choices, Harry, that show what we t..."
2,“There are only two ways to live your life. On...
3,"“The person, be it gentleman or lady, who has ..."
4,"“Imperfection is beauty, madness is genius and..."


### Convert Text to Lowercase

To ensure consistency and reduce vocabulary size, we convert all the text in the 'quotes' series to lowercase. This helps treat words like 'The' and 'the' as the same token.

In [13]:
quotes=quotes.str.lower()

### Verify Lowercasing

After converting to lowercase, we'll display the first few quotes again to confirm the transformation.

In [14]:
quotes.head()

,quote
0,“the world as we have created it is a process ...
1,"“it is our choices, harry, that show what we t..."
2,“there are only two ways to live your life. on...
3,"“the person, be it gentleman or lady, who has ..."
4,"“imperfection is beauty, madness is genius and..."


### Remove Punctuation

Punctuation marks often don't contribute to the meaning of words in a next-word prediction task and can increase vocabulary complexity. We'll remove them from the quotes using `str.maketrans` and `apply`.

In [15]:
import string
transator =  str.maketrans('','',string.punctuation)
quotes = quotes.apply(lambda x: x.translate(transator))

### Verify Punctuation Removal

Finally, we'll check the first few quotes to see the effect of punctuation removal.

In [16]:
quotes.head()

,quote
0,“the world as we have created it is a process ...
1,“it is our choices harry that show what we tru...
2,“there are only two ways to live your life one...
3,“the person be it gentleman or lady who has no...
4,“imperfection is beauty madness is genius and ...


### Import Tokenizer

We import the `Tokenizer` class from `tensorflow.keras.preprocessing.text`. This tool is essential for converting text into sequences of numbers, which is a required step before feeding text data into neural networks like LSTMs.

In [17]:
from tensorflow.keras.preprocessing.text import Tokenizer

### Initialize and Fit Tokenizer

We define a `vocab_size` (maximum number of words to keep, based on word frequency) and initialize the `Tokenizer`. We then fit the tokenizer on our cleaned `quotes` data. This step builds the vocabulary and assigns unique integer IDs to each word.

In [18]:
vocab_size = 10000

tokinizer = Tokenizer(num_words=vocab_size)
tokinizer.fit_on_texts(quotes)

### Inspect Word Index

After fitting the tokenizer, we can inspect the `word_index` attribute, which is a dictionary mapping words to their integer IDs. We'll print the total number of unique words found and the first 10 entries to verify the tokenization process.

In [19]:
word_index=tokinizer.word_index
print(len(word_index))
list(word_index.items())[:10]

8978


[('the', 1),
 ('you', 2),
 ('to', 3),
 ('and', 4),
 ('a', 5),
 ('i', 6),
 ('is', 7),
 ('of', 8),
 ('that', 9),
 ('it', 10)]

### Convert Text to Sequences

Now, we convert our text `quotes` into sequences of integers using the fitted tokenizer. Each quote becomes a list of numbers, where each number corresponds to a word in our vocabulary.

In [20]:
sequence = tokinizer.texts_to_sequences(quotes)

### Display Original Quotes

To understand the data conversion, we'll display the first three original (cleaned) quotes.

In [21]:
for i in range(3):
  print(quotes[i])

“the world as we have created it is a process of our thinking it cannot be changed without changing our thinking”
“it is our choices harry that show what we truly are far more than our abilities”
“there are only two ways to live your life one is as though nothing is a miracle the other is as though everything is a miracle”


### Display Tokenized Sequences

Next, we'll display the corresponding tokenized integer sequences for the first three quotes. This shows how each word has been mapped to a numerical ID.

In [22]:
for i in range(3):
  print(sequence[i])

[713, 62, 29, 19, 16, 946, 10, 7, 5, 1156, 8, 70, 293, 10, 145, 12, 809, 104, 752, 70, 2461]
[947, 7, 70, 871, 373, 9, 433, 21, 19, 465, 14, 294, 52, 54, 70, 3676]
[1337, 14, 53, 201, 714, 3, 81, 15, 36, 37, 7, 29, 329, 93, 7, 5, 1157, 1, 101, 7, 29, 329, 126, 7, 5, 3677]


### Create Input-Output Pairs for Next Word Prediction

For a next-word prediction model, we need to create input-output pairs. Each sequence is broken down into multiple sub-sequences. For a sequence `[w1, w2, w3, w4]`, the pairs would be:
- Input: `[w1]`, Output: `w2`
- Input: `[w1, w2]`, Output: `w3`
- Input: `[w1, w2, w3]`, Output: `w4`

This process generates `X` (input sequences) and `y` (target next words).

In [23]:
X=[]
y=[]

for seq in sequence:
  for i in range(1,len(seq)):
    input_seq = seq[:i]
    output_seq = seq[i]
    X.append(input_seq)
    y.append(output_seq)

### Check Number of Input Sequences

We'll check the total number of input sequences (`X`) generated after splitting all quotes into input-output pairs. This gives us an idea of the size of our training data.

In [24]:
len(X)

85271

### Check Number of Output Labels

Similarly, we'll check the total number of output labels (`y`). This should match the number of input sequences, as each input sequence corresponds to one target word.

In [25]:
len(y)

85271

### Determine Maximum Sequence Length

Neural networks often require fixed-size inputs. We need to find the maximum length among all input sequences in `X`. This value will be used for padding shorter sequences to ensure uniform input size for the model.

### Pad Input Sequences

To ensure all input sequences have the same length, which is required for neural network input, we'll pad the `X` sequences. We'll use pre-padding, adding zeros to the beginning of shorter sequences, up to the `max_len` determined earlier.

In [26]:
max_len=max(len(i) for i in X)
print(max_len)

745


In [27]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
X_padded = pad_sequences(X,maxlen=max_len,padding='pre')

### Convert Target Labels to NumPy Array

We convert the `y` (target words) list into a NumPy array. This is a standard practice for preparing data for TensorFlow/Keras models, which often expect NumPy arrays as input.

In [28]:
y=np.array(y)

### Check Padded Input Shape

We'll check the shape of the `X_padded` array to confirm that all sequences now have the uniform `max_len` and to see the total number of samples.

In [29]:
X_padded.shape

(85271, 745)

### One-Hot Encode Target Labels

For multi-class classification (predicting the next word from a vocabulary), the target labels (`y`) need to be one-hot encoded. This converts each integer word ID into a binary vector of size `vocab_size`.

In [30]:
from tensorflow.keras.utils import to_categorical
y_one_hot = to_categorical(y,num_classes=vocab_size)

### Check Original Target Labels Shape

We'll check the shape of the original `y` array before one-hot encoding to compare it with the one-hot encoded version and understand the transformation.

In [31]:
y.shape

(85271,)

### Check One-Hot Encoded Target Labels Shape

After one-hot encoding, we'll examine the shape of `y_one_hot`. This will show the number of samples and the size of the one-hot vector, which should be equal to our `vocab_size`.

In [32]:
y_one_hot.shape

(85271, 10000)

### Import Keras Model and Layers

We'll import the necessary components from TensorFlow Keras to define our neural network model. This includes `Sequential` for building the model layer by layer, and layers like `Embedding`, `LSTM`, and `Dense`.

In [33]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding,LSTM,Dense,SimpleRNN

### Define Model Hyperparameters

Before building the model, we define key hyperparameters such as `embedding_dim` (the dimensionality of the word embedding vectors) and `rnn_units` (the number of units in the LSTM layer). These values will influence the model's capacity and performance.

In [34]:
embedding_dim = 50
rnn_units = 128

### Define the Simple RNN Model

Here, we define a Simple Recurrent Neural Network (RNN) model using Keras's `Sequential` API. The model consists of three main layers:
1.  **Embedding Layer**: This layer converts integer-encoded words into dense vectors of fixed size (`embedding_dim`), capturing semantic relationships between words. `input_dim` is our `vocab_size`, and `input_length` is `max_len`.
2.  **SimpleRNN Layer**: This is the core recurrent layer, processing sequences. `units` specifies the dimensionality of the output space.
3.  **Dense Layer**: A final fully connected layer with a `softmax` activation function, outputting probabilities for each word in our vocabulary (`vocab_size`) as the next predicted word.

In [35]:
rnn_model =Sequential()

rnn_model.add(
Embedding(input_dim=vocab_size , output_dim= embedding_dim,input_length=max_len))

rnn_model.add(SimpleRNN(units=rnn_units))
rnn_model.add(Dense(units=vocab_size,activation='softmax'))



/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


### Compile the Simple RNN Model

Before training, we compile the RNN model. We specify:
*   **Optimizer**: `adam` is chosen for its efficiency in handling sparse gradients and adaptive learning rates.
*   **Loss Function**: `categorical_crossentropy` is appropriate for multi-class classification where target labels are one-hot encoded.
*   **Metrics**: We track `accuracy` to evaluate the model's performance during training and validation.

In [36]:
rnn_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)


### Display Simple RNN Model Summary

This cell prints a summary of the Simple RNN model's architecture. It shows the layers, their output shapes, and the number of parameters (trainable weights) for each layer, providing a concise overview of the model's complexity.

In [37]:
rnn_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

### Define the LSTM Model

Similar to the Simple RNN, we now define a Long Short-Term Memory (LSTM) model. LSTMs are a type of RNN capable of learning long-term dependencies, making them often more effective for sequence prediction tasks. The structure is similar:
1.  **Embedding Layer**: Converts word integers to dense vectors.
2.  **LSTM Layer**: The core recurrent layer, handling long-term dependencies with its gating mechanisms.
3.  **Dense Layer**: The output layer with `softmax` activation for next-word prediction probabilities.

In [38]:
lstm_model =Sequential()
lstm_model.add(Embedding(input_dim=vocab_size , output_dim =embedding_dim,
                         input_length=max_len)
)
lstm_model.add(LSTM(units=rnn_units))
lstm_model.add(Dense(units=vocab_size,activation='softmax'))

### Compile the LSTM Model

We compile the LSTM model with the same `adam` optimizer and `categorical_crossentropy` loss function as the Simple RNN. `accuracy` is again used as the evaluation metric. This step prepares the LSTM model for training.

In [39]:
lstm_model.compile(
    optimizer = 'adam',
    loss = 'categorical_crossentropy',
    metrics = ['accuracy']
)

### Display LSTM Model Summary

This cell displays a summary of the LSTM model's architecture, including layer types, output shapes, and the number of parameters. You'll often notice that LSTM layers have more parameters than Simple RNN layers due to their more complex internal structure (gates).

In [40]:
lstm_model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

### Define Training Hyperparameters

Here, we set the key hyperparameters for our model training process:
*   **`epochs`**: The number of full passes through the entire training dataset. More epochs can lead to better learning but also risk overfitting.
*   **`batch_size`**: The number of samples processed before the model's internal parameters are updated. A larger batch size means fewer updates per epoch but potentially faster computation per update.

In [41]:
epochs = 10
batch_size = 128


### Train the Simple RNN Model

We train the `rnn_model` using the prepared `X_padded` input sequences and `y_one_hot` target labels. The `fit` method orchestrates the training, performing gradient descent to minimize the loss. `validation_split` reserves a portion of the data (10%) for evaluation during training, allowing us to monitor for overfitting.

In [ ]:
history_rnn = rnn_model.fit(
    X_padded,
    y_one_hot,
    epochs=epochs,
    batch_size=batch_size,
    validation_split=0.1
)

Epoch 1/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 46s 69ms/step - accuracy: 0.0459 - loss: 6.6944 - val_accuracy: 0.0572 - val_loss: 6.5172
Epoch 2/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 38s 63ms/step - accuracy: 0.0754 - loss: 6.1004 - val_accuracy: 0.0894 - val_loss: 6.3298
Epoch 3/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 38s 64ms/step - accuracy: 0.1011 - loss: 5.7347 - val_accuracy: 0.1015 - val_loss: 6.2829
Epoch 4/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 38s 63ms/step - accuracy: 0.1184 - loss: 5.4324 - val_accuracy: 0.1034 - val_loss: 6.3440
Epoch 5/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 38s 63ms/step - accuracy: 0.1330 - loss: 5.1711 - val_accuracy: 0.1093 - val_loss: 6.3941
Epoch 6/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 38s 64ms/step - accuracy: 0.1447 - loss: 4.9352 - val_accuracy: 0.1075 - val_loss: 6.4322
Epoch 7/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 41s 63ms/step - accuracy: 0.1594 - loss: 4.7149 - val_accuracy: 0.1075 - val_loss: 6.5062
Epoch 8/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 38s 63ms/step - accuracy: 0.1756 - loss: 4.5111 - 

### Load a Pre-Trained LSTM Model

After training, models can be saved to disk. This cell demonstrates how to load a previously trained LSTM model from the file `lstm_model.h5`. This is useful for deploying models without retraining or continuing training from a checkpoint. Note: Keras might issue a warning if the compiled metrics are not explicitly rebuilt after loading.

In [42]:
from tensorflow.keras.models import load_model

lstm_model = load_model("lstm_model.h5")


### Create Index-to-Word Mapping

Our tokenizer maps words to unique integer IDs. For making predictions, the model outputs these integer IDs. To convert these IDs back into human-readable words, we need a reverse mapping. This code creates an `index_to_word` dictionary where keys are the integer IDs and values are the corresponding words, built from the `word_index` generated by the tokenizer.

In [44]:
index_to_word={}
for word, index in word_index.items():
  index_to_word[index]=word

In [48]:
# index_to_word

### Import `pad_sequences` for Prediction Preprocessing

While `pad_sequences` was used during data preparation for training, we explicitly import it again here to ensure it's available for our prediction function. This function is essential for consistently preparing new input text sequences to match the fixed input length the model was trained on.

In [45]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

### Define the Next-Word Prediction Function

This `predictor` function simplifies the process of generating the next word given a starting text. It performs the following steps:
1.  **Lowercase Input**: Converts the input `text` to lowercase for consistency.
2.  **Tokenize**: Uses the trained `tokenizer` to convert the text into a sequence of integer IDs.
3.  **Pad**: Applies `pad_sequences` to ensure the input sequence matches the `max_len` the model expects.
4.  **Predict**: Uses the `model` to get a probability distribution over the vocabulary for the next word.
5.  **Get Predicted Index**: Finds the word index with the highest probability using `np.argmax`.
6.  **Convert to Word**: Maps the predicted index back to a word using the `index_to_word` dictionary and returns it.

In [50]:
def predictor(model,tokinizer,text,max_len):
  text = text.lower()

  seq=tokinizer.texts_to_sequences([text])[0]
  seq=pad_sequences([seq],maxlen=max_len,padding='pre')

  pred=model.predict(seq,verbose=0)
  pred_index = np.argmax(pred)
  return index_to_word[pred_index]

### Test the Next-Word Predictor

Now we can test our `predictor` function. We provide a `seed_text` (e.g., 'what are you') and the function will return the model's predicted next word based on the patterns it learned during training.

In [53]:
seed_text= "what are you"
next_word= predictor(lstm_model,tokinizer,seed_text,max_len)
print(next_word)

worrying


### Define the Text Generation Function

To generate a longer sequence of text, we create a `generate_text` function. This function iteratively calls the `predictor` function multiple times, appending each predicted word to the `seed_text` until a specified number of words (`n_words`) is generated, or if the predictor returns an empty string (which can happen if a word is not found in `index_to_word`).

In [62]:
def generate_text(model,tokinizer,seed_text,max_len,n_words):
  for _ in range(n_words):
    next_word = predictor(model,tokinizer,seed_text,max_len)
    if next_word == "":
      break
    seed_text += " " + next_word
  return seed_text

### Generate a Sequence of Text

Here, we use our `generate_text` function to create a longer passage. We provide an initial `seed` text and specify how many `n_words` we want the model to generate. The generated output is then printed.

In [66]:
seed= "are you "
generated_text = generate_text(lstm_model,tokinizer,seed,max_len,10)
print(generated_text)

are you  implying that shreds of my reputation remain intact will demanded


### Save the Tokenizer

To ensure reproducibility and to avoid re-fitting the tokenizer every time we use the model, we save the `tokinizer` object using Python's `pickle` module. This allows us to load the exact same vocabulary mapping later when making predictions in a new session or deployment.

In [69]:
import pickle
with open("tokenizer.pkl","wb") as f:
  pickle.dump(tokinizer,f)

### Save the Maximum Sequence Length

Similar to the tokenizer, the `max_len` (maximum sequence length) used for padding is a crucial parameter for our model. We save this value using `pickle` so that when we load the model for inference, we can correctly pad new input sequences to the length the model expects.

In [68]:
with open ("max_len.pkl","wb") as f:
  pickle.dump(max_len,f)